In [1]:
#import numpy as np
import pandas as pd

#from sklearn.model_selection import train_test_split
#from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
#from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

import mlflow

In [2]:
data = pd.read_csv("../data/processed/transfer_modeling_dataset.csv")
print(data['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    82.801498
1    17.198502
Name: proportion, dtype: float64


In [3]:
NON_FEATURE_COLUMNS = {
    "transfer_key",
    "player_id",
    "player_full_name",
    "player_name",
    "source_club_id",
    "destination_club_id",
    "source_club_name",
    "destination_club_name",
    "source_club_dataset_name",
    "destination_club_dataset_name",
    "follow_up_window_end",
    "pre_transfer_market_value_date",
    "market_value_180d_prior_date",
    "market_value_365d_prior_date",
    "target_end_market_value_date",
    "target_failure_reason",
    "target_is_eligible",
    "exclusion_reason",
    "date_of_birth",
    "source_country_name",
    "destination_country_name",
    "pre_transfer_market_value_club_id",
    "pre_transfer_market_value_league_id",
    "valuation_cutoff_180d",
    "valuation_cutoff_365d",
    "player_window_start_180d",
    "player_window_end_180d",
    "player_window_start_365d",
    "player_window_end_365d",
    "source_api_cache_file",
    "destination_api_cache_file",
    "player_current_market_value",
    "highest_market_value_in_eur",
    "transfer_season",
    "source_competition_name",
    "destination_competition_name",
    "source_competition_name",
    "destination_competition_name",
    "target_destination_matches_24m",
    "target_destination_minutes_24m",
    "target_destination_goals_24m",
    "target_destination_assists_24m",
    "target_end_market_value",
    "target_market_value_delta_24m"
}
print(len(NON_FEATURE_COLUMNS))
data.drop(columns=NON_FEATURE_COLUMNS, inplace=True)

42


In [4]:
# 1. Select the row (e.g., the first row at index 0)
row_data = data.iloc[0]

# 2. Combine values and data types into a single table
result = pd.DataFrame({
    'Value': row_data,
    'DataType': data.dtypes
})

print(result)


                                      Value DataType
transfer_date                    2018-07-01   object
source_competition_id                   IT1   object
destination_competition_id              IT1   object
age_at_transfer                        35.0  float64
position                         Goalkeeper   object
...                                     ...      ...
destination_api_context_matched           0    int64
is_same_league_move                       1    int64
is_same_country_move                      1    int64
season_start_year                      2018    int64
transfer_success                          0    int64

[61 rows x 2 columns]


In [5]:
# Returns a list of column names
cols_with_nan = data.columns[data.isna().any()].tolist()
print(cols_with_nan)
print(len(cols_with_nan))

['source_competition_id', 'age_at_transfer', 'foot', 'transfer_fee', 'market_value_in_eur', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_goals_per90_180d_pre', 'player_assists_per90_180d_pre', 'player_goals_per90_365d_pre', 'player_assists_per90_365d_pre', 'source_squad_size', 'source_average_age', 'source_foreigners_percentage', 'source_national_team_players', 'source_stadium_seats', 'destination_average_age', 'destination_foreigners_percentage', 'source_win_rate_365d_pre', 'source_points_per_match_365d_pre', 'source_goal_diff_per_match_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre', 'source_api_cached_matches', 'source_api_cached_win_rate', 'source_api_cached_goal_diff_per_match', 'destination_api_cached_matches', 'destination_api_cached_win_rate', 'destination

In [6]:
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))

6675


In [ ]:
# 1. Define your threshold
threshold = 0.2*len(data)  # For example, 20% of the total number of rows

# 2. Identify columns with NaNs > threshold
# data.isna().sum() returns a Series where index is column name and value is NaN count
nan_counts = data.isna().sum()
cols_to_drop = nan_counts[nan_counts > threshold]

# 3. Print or Log the results
if not cols_to_drop.empty:
    print(f"Columns with more than {threshold} NaNs:")
    print(cols_to_drop)
else:
    print("No columns exceed the NaN threshold.")

Columns with more than 1335.0 NaNs:
source_competition_id                         1501
player_goals_per90_180d_pre                   3048
player_assists_per90_180d_pre                 3048
player_goals_per90_365d_pre                   2485
player_assists_per90_365d_pre                 2485
source_squad_size                             1501
source_average_age                            1532
source_foreigners_percentage                  1538
source_national_team_players                  1501
source_stadium_seats                          1501
source_win_rate_365d_pre                      1527
source_points_per_match_365d_pre              1527
source_goal_diff_per_match_365d_pre           1527
source_api_cached_matches                     6675
source_api_cached_win_rate                    6675
source_api_cached_goal_diff_per_match         6675
destination_api_cached_matches                6674
destination_api_cached_win_rate               6674
destination_api_cached_goal_diff_per_match    

In [8]:
data = data.dropna(axis=1,thresh=0.8*len(data))

In [9]:
# Returns a Series with column names and their respective NaN counts
nan_counts = data.isna().sum()

# Filter to show only columns that have more than 0 NaNs
print(nan_counts[nan_counts > 0])

age_at_transfer                               2
foot                                         12
transfer_fee                                559
market_value_in_eur                           5
market_value_180d_prior                     226
market_value_365d_prior                     539
market_value_change_180d                    226
market_value_change_365d                    539
transfer_fee_to_market_value_ratio          559
transfer_fee_minus_market_value             559
destination_average_age                      23
destination_foreigners_percentage            23
destination_win_rate_365d_pre               314
destination_points_per_match_365d_pre       314
destination_goal_diff_per_match_365d_pre    314
dtype: int64


In [10]:
metadata = {
    'features': data.columns.tolist(),
    'data_types': data.dtypes.tolist(),
}
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("../data/metadata.csv", index=False)

In [11]:
print(len(data.columns))
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))
print(len(rows_with_nan)/len(data))

42
1135
0.1700374531835206


In [12]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
                                  Column Data Type  NaN Count  % Missing
                         age_at_transfer   float64          2       0.03
                                    foot    object         12       0.18
                            transfer_fee   float64        559       8.37
                     market_value_in_eur   float64          5       0.07
                 market_value_180d_prior   float64        226       3.39
                 market_value_365d_prior   float64        539       8.07
                market_value_change_180d   float64        226       3.39
                market_value_change_365d   float64        539       8.07
      transfer_fee_to_market_value_ratio   float64        559       8.37
         transfer_fee_minus_market_value   float64        559       8.37
                 destination_average_age   float64         23       0.34
       destination_foreigners_percentage   float64         23       0.34
           

In [13]:
data['age_at_transfer'].fillna(data['age_at_transfer'].median(), inplace=True)
data['foot'].fillna(data['foot'].mode()[0], inplace=True)
data['transfer_fee'].fillna(data['transfer_fee'].median(), inplace=True)
data['market_value_in_eur'].fillna(data['market_value_in_eur'].median(), inplace=True)
data['market_value_180d_prior'].fillna(data['market_value_180d_prior'].median(), inplace=True)
data['market_value_365d_prior'].fillna(data['market_value_365d_prior'].median(), inplace=True)
data['market_value_change_180d'].fillna(data['market_value_change_180d'].median(), inplace=True)
data['market_value_change_365d'].fillna(data['market_value_change_365d'].median(), inplace=True)
data['transfer_fee_to_market_value_ratio'].fillna(data['transfer_fee_to_market_value_ratio'].median(), inplace=True)
data['transfer_fee_minus_market_value'].fillna(data['transfer_fee_minus_market_value'].median(), inplace=True)
data['destination_average_age'].fillna(data['destination_average_age'].median(), inplace=True)
data['destination_foreigners_percentage'].fillna(data['destination_foreigners_percentage'].median(), inplace=True)
data['destination_win_rate_365d_pre'].fillna(data['destination_win_rate_365d_pre'].median(), inplace=True)
data['destination_points_per_match_365d_pre'].fillna(data['destination_points_per_match_365d_pre'].median(), inplace=True)
data['destination_goal_diff_per_match_365d_pre'].fillna(data['destination_goal_diff_per_match_365d_pre'].median(), inplace=True)

C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_15636\1913183770.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['age_at_transfer'].fillna(data['age_at_transfer'].median(), inplace=True)
C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_15636\1913183770.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

In [14]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
Empty DataFrame
Columns: [Column, Data Type, NaN Count, % Missing]
Index: []


In [15]:
import sys 
from pathlib import Path
ROOT =Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.features.build_features import chronological_split
data_dict = chronological_split(data)
#train_data, test_data = train_test_split(data, test_size=0.15, shuffle=True, random_state=42)
print(data_dict['train']['transfer_success'].value_counts(normalize=True)*100)
print(data_dict['test']['transfer_success'].value_counts(normalize=True)*100)
data_dict['train'].to_csv("../data/model/train_data.csv", index=False)
data_dict['test'].to_csv("../data/model/test_data.csv", index=False) 

transfer_success
0    81.870719
1    18.129281
Name: proportion, dtype: float64
transfer_success
0    90.419162
1     9.580838
Name: proportion, dtype: float64


In [16]:
# I will use simple validation technique just divide train data into train and validation
#train_data, val_data = train_test_split(train_data, test_size=0.15, shuffle=True, random_state=42)
print(data_dict['train']['transfer_success'].value_counts(normalize=True)*100)
print(data_dict['val']['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    81.870719
1    18.129281
Name: proportion, dtype: float64
transfer_success
0    79.52048
1    20.47952
Name: proportion, dtype: float64


In [26]:
y_train =   data_dict['train']['transfer_success']
X_train = data_dict['train'].drop(columns=['transfer_success'])
y_val =   data_dict['val']['transfer_success']
X_val = data_dict['val'].drop(columns=['transfer_success'])

In [18]:
# 1. Get the count of each data type
type_counts = data_dict['train'].dtypes.value_counts()

print("--- Data Type Counts ---")
print(type_counts)

# 2. See exactly which columns belong to each type
print("\n--- Columns by Type ---")
for dtype in type_counts.index:
    cols = data_dict['train'].select_dtypes(include=[dtype]).columns.tolist()
    print(f"{dtype}: {cols}")

--- Data Type Counts ---
float64    26
int64       9
object      7
Name: count, dtype: int64

--- Columns by Type ---
float64: ['age_at_transfer', 'height_in_cm', 'transfer_fee', 'market_value_in_eur', 'pre_transfer_market_value', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_matches_180d_pre', 'player_minutes_180d_pre', 'player_goals_180d_pre', 'player_assists_180d_pre', 'player_matches_365d_pre', 'player_minutes_365d_pre', 'player_goals_365d_pre', 'player_assists_365d_pre', 'destination_average_age', 'destination_foreigners_percentage', 'source_matches_365d_pre', 'destination_matches_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre']
int64: ['destination_squad_size', 'destination_national_team_players', 'destination_stadium_seats', 'source_api_context_matched', 'de

In [27]:
# now I will handle categorical features with frequency encoding as sklearn does not support categorical features
# directly and I want to avoid one-hot encoding due to high cardinality of some features.
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    freq_encoding = X_train[col].value_counts() / len(X_train)
    X_train[col] = X_train[col].map(freq_encoding)
    X_val[col] = X_val[col].map(freq_encoding).fillna(0)  # Handle unseen categories in validation set
    data_dict['test'][col] = data_dict['test'][col].map(freq_encoding).fillna(0)  # Handle unseen categories in test set

data_dict['test'].to_csv("../data/model/test_data_encoded.csv", index=False)


In [28]:

experiment = mlflow.set_experiment("Training Models")
experiment_id = experiment.experiment_id
#mlflow.sklearn.autolog()
results_train_df = pd.DataFrame(columns=["model_name", "accuracy", "precision", "recall", "f1_score","run_id"])
results_val_df = pd.DataFrame(columns=["model_name", "accuracy", "precision", "recall", "f1_score","run_id"])

# first I will try dummy classifier to set a baseline for the model performance
with mlflow.start_run(run_name="ZeroR Classifier", experiment_id=experiment_id) as run:
    ZeroR = DummyClassifier(strategy='most_frequent')
    ZeroR.fit(X_train, y_train)
    y_pred_val = ZeroR.predict(X_val)
    y_pred_train = ZeroR.predict(X_train)
    print("ZeroR Classifier")
    mlflow.log_metric("val accuracy", accuracy_score(y_val, y_pred_val))
    mlflow.log_metric("val precision", precision_score(y_val, y_pred_val))
    mlflow.log_metric("val recall", recall_score(y_val, y_pred_val))
    mlflow.log_metric("val f1_score", f1_score(y_val, y_pred_val))
    mlflow.log_metric("train accuracy", accuracy_score(y_train, y_pred_train))
    mlflow.log_metric("train precision", precision_score(y_train, y_pred_train))
    mlflow.log_metric("train recall", recall_score(y_train, y_pred_train))
    mlflow.log_metric("train f1_score", f1_score(y_train, y_pred_train))
    mlflow.sklearn.log_model(sk_model=ZeroR, name="zero_r_classifier")
    # Append the results to the DataFrame
    results_val_df = pd.concat([results_val_df, pd.DataFrame([{
        "model_name": "ZeroR Classifier",
        "accuracy": accuracy_score(y_val, y_pred_val),
        "precision": precision_score(y_val, y_pred_val),
        "recall": recall_score(y_val, y_pred_val),
        "f1_score": f1_score(y_val, y_pred_val),
        "run_id": run.info.run_id
    }])])

    results_train_df = pd.concat([results_train_df, pd.DataFrame([{
        "model_name": "ZeroR Classifier",
        "accuracy": accuracy_score(y_train, y_pred_train),
        "precision": precision_score(y_train, y_pred_train),
        "recall": recall_score(y_train, y_pred_train),
        "f1_score": f1_score(y_train, y_pred_train),
        "run_id": run.info.run_id
    }])])


ZeroR Classifier


c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/05/03 21:00:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code durin

In [29]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=0.5,k_neighbors=5, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(y_train.value_counts(normalize=True)*100)

transfer_success
0    66.672477
1    33.327523
Name: proportion, dtype: float64


In [30]:
# now I will use decision tree classifier
hyperparameters = [5, 10, 15, 20]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Decision Tree min_samples_split={h}", experiment_id=experiment_id) as run:
        params = {
            'criterion': 'entropy',
            'max_depth': 41,
            'min_samples_split': h,
        }
        mlflow.log_params(params)
        dt = DecisionTreeClassifier(random_state=42, **params)
        dt.fit(X_train, y_train)
        y_pred_val = dt.predict(X_val)
        y_pred_train = dt.predict(X_train)
        print("Decision Tree Classifier for min_samples_split =", h)
        mlflow.log_metric("val accuracy", accuracy_score(y_val, y_pred_val))
        mlflow.log_metric("val precision", precision_score(y_val, y_pred_val))
        mlflow.log_metric("val recall", recall_score(y_val, y_pred_val))
        mlflow.log_metric("val f1_score", f1_score(y_val, y_pred_val))
        mlflow.log_metric("train accuracy", accuracy_score(y_train, y_pred_train))
        mlflow.log_metric("train precision", precision_score(y_train, y_pred_train))
        mlflow.log_metric("train recall", recall_score(y_train, y_pred_train))
        mlflow.log_metric("train f1_score", f1_score(y_train, y_pred_train))
        mlflow.sklearn.log_model(sk_model=dt, name=f"decision_tree_min_samples_split_{h}")
        results_val_df = pd.concat([results_val_df, pd.DataFrame([{
            "model_name": f"Decision Tree min_samples_split={h}",
            "accuracy": accuracy_score(y_val, y_pred_val),
            "precision": precision_score(y_val, y_pred_val),
            "recall": recall_score(y_val, y_pred_val),
            "f1_score": f1_score(y_val, y_pred_val),
            "run_id": run.info.run_id
        }])])
        results_train_df = pd.concat([results_train_df, pd.DataFrame([{
            "model_name": f"Decision Tree min_samples_split={h}",
            "accuracy": accuracy_score(y_train, y_pred_train),
            "precision": precision_score(y_train, y_pred_train),
            "recall": recall_score(y_train, y_pred_train),
            "f1_score": f1_score(y_train, y_pred_train),
            "run_id": run.info.run_id
        }])])

Decision Tree Classifier for min_samples_split = 5


2026/05/03 21:05:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree Classifier for min_samples_split = 10


2026/05/03 21:05:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree Classifier for min_samples_split = 15


2026/05/03 21:05:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree Classifier for min_samples_split = 20


2026/05/03 21:05:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [31]:
# now trying logistic regression
hyperparameters = [10,100,1000]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Logistic Regression max_iter={h}", experiment_id=experiment_id) as run:
        params = {
            'C': 1.0,
            'max_iter': h,
            'solver': 'lbfgs',
            'random_state': 42
        }
        mlflow.log_params(params)
        lr = LogisticRegression(**params)
        lr.fit(X_train,y_train)
        y_pred_val = lr.predict(X_val)
        y_pred_train = lr.predict(X_train)
        print("LogisticRegression for max_iter =", h)
        mlflow.log_metric("val accuracy", accuracy_score(y_val, y_pred_val))
        mlflow.log_metric("val precision", precision_score(y_val, y_pred_val))
        mlflow.log_metric("val recall", recall_score(y_val, y_pred_val))
        mlflow.log_metric("val f1_score", f1_score(y_val, y_pred_val))
        mlflow.log_metric("train accuracy", accuracy_score(y_train, y_pred_train))
        mlflow.log_metric("train precision", precision_score(y_train, y_pred_train))
        mlflow.log_metric("train recall", recall_score(y_train, y_pred_train))
        mlflow.log_metric("train f1_score", f1_score(y_train, y_pred_train))
        mlflow.sklearn.log_model(sk_model=lr, name=f"logistic_regression_max_iter_{h}")
        results_val_df = pd.concat([results_val_df, pd.DataFrame([{
            "model_name": f"Logistic Regression max_iter={h}",
            "accuracy": accuracy_score(y_val, y_pred_val),
            "precision": precision_score(y_val, y_pred_val),
            "recall": recall_score(y_val, y_pred_val),
            "f1_score": f1_score(y_val, y_pred_val),
            "run_id": run.info.run_id
        }])])
        results_train_df = pd.concat([results_train_df, pd.DataFrame([{
            "model_name": f"Logistic Regression max_iter={h}",
            "accuracy": accuracy_score(y_train, y_pred_train),
            "precision": precision_score(y_train, y_pred_train),
            "recall": recall_score(y_train, y_pred_train),
            "f1_score": f1_score(y_train, y_pred_train),
            "run_id": run.info.run_id
        }])])

c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression for max_iter = 10


2026/05/03 21:08:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logist

LogisticRegression for max_iter = 100


2026/05/03 21:08:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logi

LogisticRegression for max_iter = 1000


2026/05/03 21:08:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [32]:
# now trying linear svm model
hyperparameters = [1,10,50,100]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Linear SVM C={h}", experiment_id=experiment_id) as run:
        params = {
            'C': h,
            'random_state': 42
        }
        mlflow.log_params(params)
        model_svm = svm.LinearSVC(**params)
        model_svm.fit(X_train,y_train)
        y_val_pred = model_svm.predict(X_val)
        y_train_pred = model_svm.predict(X_train)
        print("Linear SVM for C =", h)
        mlflow.log_metric("val accuracy", accuracy_score(y_val, y_val_pred))
        mlflow.log_metric("val precision", precision_score(y_val, y_val_pred))
        mlflow.log_metric("val recall", recall_score(y_val, y_val_pred))
        mlflow.log_metric("val f1_score", f1_score(y_val, y_val_pred))
        mlflow.log_metric("train accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train f1_score", f1_score(y_train, y_train_pred))
        mlflow.sklearn.log_model(sk_model=model_svm, name=f"linear_svm_C_{h}")
        results_val_df = pd.concat([results_val_df, pd.DataFrame([{
            "model_name": f"Linear SVM C={h}",
            "accuracy": accuracy_score(y_val, y_val_pred),
            "precision": precision_score(y_val, y_val_pred),
            "recall": recall_score(y_val, y_val_pred),
            "f1_score": f1_score(y_val, y_val_pred),
            "run_id": run.info.run_id    
        }])])
        results_train_df = pd.concat([results_train_df, pd.DataFrame([{
            "model_name": f"Linear SVM C={h}",
            "accuracy": accuracy_score(y_train, y_train_pred),
            "precision": precision_score(y_train, y_train_pred),
            "recall": recall_score(y_train, y_train_pred),
            "f1_score": f1_score(y_train, y_train_pred),
            "run_id": run.info.run_id
        }])])

Linear SVM for C = 1


2026/05/03 21:10:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear SVM for C = 10


2026/05/03 21:10:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear SVM for C = 50


2026/05/03 21:11:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear SVM for C = 100


2026/05/03 21:11:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [33]:
# now tring random forest classifier
hyperparameters = [5,10,15,20]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Random Forest n_estimators={h}", experiment_id=experiment_id) as run:
        params = {
            'n_estimators': h,
            'random_state': 42
        }
        mlflow.log_params(params)
        model_rf = RandomForestClassifier(**params)
        model_rf.fit(X_train,y_train)
        y_val_pred = model_rf.predict(X_val)
        y_train_pred = model_rf.predict(X_train)
        print("Random Forest Classifier for n_estimators =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_val_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_val_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_val_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_val_pred))
        mlflow.log_metric("train accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train f1_score", f1_score(y_train, y_train_pred))
        mlflow.sklearn.log_model(sk_model=model_rf, name=f"random_forest_n_estimators_{h}")
        results_val_df = pd.concat([results_val_df, pd.DataFrame([{
            "model_name": f"Random Forest n_estimators={h}",
            "accuracy": accuracy_score(y_val, y_val_pred),
            "precision": precision_score(y_val, y_val_pred),
            "recall": recall_score(y_val, y_val_pred),
            "f1_score": f1_score(y_val, y_val_pred),
            "run_id": run.info.run_id
        }])])
        results_train_df = pd.concat([results_train_df, pd.DataFrame([{
            "model_name": f"Random Forest n_estimators={h}",
            "accuracy": accuracy_score(y_train, y_train_pred),
            "precision": precision_score(y_train, y_train_pred),
            "recall": recall_score(y_train, y_train_pred),
            "f1_score": f1_score(y_train, y_train_pred),
            "run_id": run.info.run_id
        }])])

Random Forest Classifier for n_estimators = 5


2026/05/03 21:15:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Classifier for n_estimators = 10


2026/05/03 21:15:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Classifier for n_estimators = 15


2026/05/03 21:15:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Classifier for n_estimators = 20


2026/05/03 21:15:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [35]:
# now trying XGBoost classifier
hyperparameters = [5,10,20,50]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"XGBoost n_estimators={h}", experiment_id=experiment_id) as run:
        params = {
            'n_estimators': h,
            'max_depth': 41,
            'learning_rate': 0.1,
        }
        mlflow.log_params(params)
        xgb = XGBClassifier(**params)
        xgb.fit(X_train, y_train)
        y_pred_val = xgb.predict(X_val)
        y_pred_train = xgb.predict(X_train)
        print("XGBoost Classifier for n_estimators =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred_val))
        mlflow.log_metric("precision", precision_score(y_val, y_pred_val))
        mlflow.log_metric("recall", recall_score(y_val, y_pred_val))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred_val))
        mlflow.log_metric("train accuracy", accuracy_score(y_train, y_pred_train))
        mlflow.log_metric("train precision", precision_score(y_train, y_pred_train))
        mlflow.log_metric("train recall", recall_score(y_train, y_pred_train))
        mlflow.log_metric("train f1_score", f1_score(y_train, y_pred_train))
        mlflow.xgboost.log_model(xgb_model=xgb, name=f"xgboost_n_estimators_{h}")
        results_val_df = pd.concat([results_val_df, pd.DataFrame([{
            "model_name": f"XGBoost n_estimators={h}",
            "accuracy": accuracy_score(y_val, y_pred_val),
            "precision": precision_score(y_val, y_pred_val),
            "recall": recall_score(y_val, y_pred_val),
            "f1_score": f1_score(y_val, y_pred_val),
            "run_id": run.info.run_id
        }])])
        results_train_df = pd.concat([results_train_df, pd.DataFrame([{
            "model_name": f"XGBoost n_estimators={h}",
            "accuracy": accuracy_score(y_train, y_pred_train),
            "precision": precision_score(y_train, y_pred_train),
            "recall": recall_score(y_train, y_pred_train),
            "f1_score": f1_score(y_train, y_pred_train),
            "run_id": run.info.run_id
        }])])

XGBoost Classifier for n_estimators = 5
XGBoost Classifier for n_estimators = 10
XGBoost Classifier for n_estimators = 20
XGBoost Classifier for n_estimators = 50


In [37]:
print(results_val_df)
print(results_train_df)

                           model_name  accuracy  precision    recall  \
0                    ZeroR Classifier  0.795205   0.000000  0.000000   
0   Decision Tree min_samples_split=5  0.742258   0.364103  0.346341   
0  Decision Tree min_samples_split=10  0.748252   0.368715  0.321951   
0  Decision Tree min_samples_split=15  0.762238   0.402367  0.331707   
0  Decision Tree min_samples_split=20  0.751249   0.379121  0.336585   
0     Logistic Regression max_iter=10  0.771229   0.424051  0.326829   
0    Logistic Regression max_iter=100  0.793207   0.487500  0.190244   
0   Logistic Regression max_iter=1000  0.792208   0.479452  0.170732   
0                      Linear SVM C=1  0.793207   0.484375  0.151220   
0                     Linear SVM C=10  0.795205   0.500000  0.156098   
0                     Linear SVM C=50  0.793207   0.484375  0.151220   
0                    Linear SVM C=100  0.796204   0.507937  0.156098   
0        Random Forest n_estimators=5  0.766234   0.392593  0.25

In [ ]:
from docx import Document

# 1. Prepare your DataFrame
# Dropping 'run_id' if it exists (errors='ignore' prevents a crash if it's missing)
df_filtered = results_test_df.drop(columns=['run_id'], errors='ignore')

# 2. Round numerical columns to 6 decimal places
df_final = df_filtered.round(6)

# 3. Initialize Word Document
doc = Document()
doc.add_heading('Refined Model Results', level=1)

# 4. Create and Fill the Table
table = doc.add_table(rows=(df_final.shape[0] + 1), cols=df_final.shape[1])
table.style = 'Table Grid'

# Set Header Row
for j in range(df_final.shape[1]):
    table.cell(0, j).text = str(df_final.columns[j])

# Set Data Rows
for i in range(df_final.shape[0]):
    for j in range(df_final.shape[1]):
        # The str() conversion ensures rounded floats are written correctly
        table.cell(i + 1, j).text = str(df_final.values[i, j])

doc.save('Cleaned_Results.docx')

In [46]:
y_test = data_dict['test']['transfer_success']
X_test = data_dict['test'].drop(columns=['transfer_success'])

In [47]:
results_test_df = pd.DataFrame(columns=["model_name", "accuracy", "precision", "recall", "f1_score","run_id"])
# Load baseline model
run_id = 'f57f4c6eaf4c4941996d85a95dff54da'
model_name = 'zero_r_classifier'
model_uri = f'runs:/{run_id}/{model_name}'
zeroR = mlflow.sklearn.load_model(model_uri=model_uri)
y_pred = zeroR.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "ZeroR Classifier",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_15636\2834300546.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_test_df = pd.concat([results_test_df, pd.DataFrame([{


In [48]:
# Load best decision tree model
run_id = 'e2e975fce8d24cc78399e81bbaa9f7eb'
model_name = 'decision_tree_min_samples_split_15'
model_uri = f'runs:/{run_id}/{model_name}'
dt = mlflow.sklearn.load_model(model_uri=model_uri)
y_pred = dt.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "Decision Tree min_samples_split=15",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

In [49]:
# Load best logistic regression model
run_id = '3f3f5f97f990489194867fa3cd12e71c'
model_name = 'logistic_regression_max_iter_10'
model_uri = f'runs:/{run_id}/{model_name}'
lr = mlflow.sklearn.load_model(model_uri=model_uri)
y_pred = lr.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "Logistic Regression max_iter=10",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

In [50]:
# Load best linear svm model
run_id = '721f9938b35e4afa8bff38210fe8ca61'
model_name = 'linear_svm_C_100'
model_uri = f'runs:/{run_id}/{model_name}'
model_svm = mlflow.sklearn.load_model(model_uri=model_uri)
y_pred = model_svm.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "Linear SVM C=100",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

In [51]:
# load best random forest model
run_id = 'c1c82e6e8f264e3d8e93ff3a9baa9239'
model_name = 'random_forest_n_estimators_5'
model_uri = f'runs:/{run_id}/{model_name}'
model_rf = mlflow.sklearn.load_model(model_uri=model_uri)
y_pred = model_rf.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "Random Forest n_estimators=5",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

In [53]:
#load best xgboost model
run_id = '31620986e41440b480ee7c49087d48ee'
model_name = 'xgboost_n_estimators_50'
model_uri = f'runs:/{run_id}/{model_name}'
xgb = mlflow.xgboost.load_model(model_uri=model_uri)
y_pred = xgb.predict(X_test)
results_test_df = pd.concat([results_test_df, pd.DataFrame([{
    "model_name": "XGBoost n_estimators=50",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "run_id": run_id
}])])

In [54]:
print(results_test_df)

                           model_name  accuracy  precision    recall  \
0                    ZeroR Classifier  0.904192   0.000000  0.000000   
0  Decision Tree min_samples_split=15  0.694611   0.116788  0.333333   
0     Logistic Regression max_iter=10  0.893214   0.406780  0.250000   
0                    Linear SVM C=100  0.904192   0.500000  0.104167   
0        Random Forest n_estimators=5  0.834331   0.169811  0.187500   
0             XGBoost n_estimators=50  0.901198   0.421053  0.083333   

   f1_score                            run_id  
0  0.000000  f57f4c6eaf4c4941996d85a95dff54da  
0  0.172973  e2e975fce8d24cc78399e81bbaa9f7eb  
0  0.309677  3f3f5f97f990489194867fa3cd12e71c  
0  0.172414  721f9938b35e4afa8bff38210fe8ca61  
0  0.178218  c1c82e6e8f264e3d8e93ff3a9baa9239  
0  0.139130  31620986e41440b480ee7c49087d48ee  


In [55]:
from docx import Document

# 1. Prepare your DataFrame
# Dropping 'run_id' if it exists (errors='ignore' prevents a crash if it's missing)
df_filtered = results_test_df.drop(columns=['run_id'], errors='ignore')

# 2. Round numerical columns to 6 decimal places
df_final = df_filtered.round(6)

# 3. Initialize Word Document
doc = Document()
doc.add_heading('Refined Model Results', level=1)

# 4. Create and Fill the Table
table = doc.add_table(rows=(df_final.shape[0] + 1), cols=df_final.shape[1])
table.style = 'Table Grid'

# Set Header Row
for j in range(df_final.shape[1]):
    table.cell(0, j).text = str(df_final.columns[j])

# Set Data Rows
for i in range(df_final.shape[0]):
    for j in range(df_final.shape[1]):
        # The str() conversion ensures rounded floats are written correctly
        table.cell(i + 1, j).text = str(df_final.values[i, j])

doc.save('Cleaned_Results.docx')